<a href="https://colab.research.google.com/github/julipolovinkina-hub/OOP_1/blob/main/OOP_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import List

# ==========================================
# БАЗОВЫЙ КЛАСС 1 И ЕГО ПОТОМКИ
# ==========================================

class MenuItem:
    """Базовый класс для любой позиции в меню ресторана."""

    def __init__(self, name: str, price: float):
        self.name = name
        self.price = price

    def apply_discount(self, is_loyal: bool) -> float:
        """
        Полиморфный метод: рассчитывает сумму скидки.
        По умолчанию скидка отсутствует.
        """
        return 0.0

    def __str__(self) -> str:
        return f"{self.name} | Цена: {self.price:.2f} руб."


class Dish(MenuItem):
    """Класс блюда, наследуемый от MenuItem."""

    def __init__(self, name: str, price: float, ingredients: str):
        super().__init__(name, price)
        self.ingredients = ingredients

    def apply_discount(self, is_loyal: bool) -> float:
        """
        Переопределение: держатели карт лояльности получают скидку 15% на блюда.
        """
        if is_loyal:
            return self.price * 0.15
        return 0.0

    def __str__(self) -> str:
        base_str = super().__str__()
        return f"{base_str} | Состав: {self.ingredients}"


class Drink(MenuItem):
    """Класс напитка, наследуемый от MenuItem."""

    def __init__(self, name: str, price: float, volume_ml: int):
        super().__init__(name, price)
        self.volume_ml = volume_ml

    def apply_discount(self, is_loyal: bool) -> float:
        """
        Переопределение: на напитки скидка не распространяется (возврат 0.0).
        """
        return 0.0

    def __str__(self) -> str:
        base_str = super().__str__()
        return f"{base_str} | Объем: {self.volume_ml} мл"


# ==========================================
# БАЗОВЫЙ КЛАСС 2 И ЕГО ПОТОМОК
# ==========================================

class Customer:
    """Базовый класс посетителя ресторана."""

    def __init__(self, name: str, table_number: int):
        self.name = name
        self.table_number = table_number

    @property
    def is_loyal(self) -> bool:
        """Свойство для полиморфной проверки статуса лояльности."""
        return False

    def __str__(self) -> str:
        return f"Гость: {self.name} | Столик: №{self.table_number}"


class LoyaltyCardHolder(Customer):
    """Класс гостя с картой лояльности, наследуемый от Customer."""

    def __init__(self, name: str, table_number: int, points: int):
        super().__init__(name, table_number)
        self.points = points

    @property
    def is_loyal(self) -> bool:
        """Переопределение свойства: гость является лояльным."""
        return True

    def __str__(self) -> str:
        base_str = super().__str__()
        return f"{base_str} | Баллы лояльности: {self.points}"


# ==========================================
# КЛАСС-КОНТЕЙНЕР
# ==========================================

class Check:
    """Класс чека, агрегирующий клиента и список заказанных позиций."""

    def __init__(self, customer: Customer, items: List[MenuItem]):
        self.customer = customer
        self.items = items

    def calculate_total(self) -> float:
        """
        Рассчитывает итоговую сумму чека, применяя полиморфный метод
        apply_discount() для каждой позиции в зависимости от статуса клиента.
        """
        total = 0.0
        is_loyal = self.customer.is_loyal

        for item in self.items:
            discount = item.apply_discount(is_loyal)
            final_price = item.price - discount
            total += final_price

        return total

    def __str__(self) -> str:
        return f"Чек для: {self.customer.name} | Позиций: {len(self.items)} | Итого: {self.calculate_total():.2f} руб."

In [ ]:
def main() -> None:
    """Демонстрационный сценарий работы системы ресторана."""

    print("=" * 70)
    print("ДОБРО ПОЖАЛОВАТЬ В РЕСТОРАН 'PYTHON GOURMET'")
    print("=" * 70)

    # 1. Создаем объекты меню (разные типы)
    print("\n--- Формирование меню ---")
    dish_pasta = Dish(name="Паста Карбонара", price=650.0, ingredients="Спагетти, бекон, сливки, пармезан")
    dish_steak = Dish(name="Стейк Рибай", price=1800.0, ingredients="Говядина, розмарин, чеснок")
    drink_wine = Drink(name="Вино красное", price=450.0, volume_ml=150)
    drink_water = Drink(name="Вода минеральная", price=150.0, volume_ml=500)

    menu_items = [dish_pasta, dish_steak, drink_wine, drink_water]
    for item in menu_items:
        print(item)

    # 2. Создаем клиентов (обычный и лояльный)
    print("\n--- Регистрация гостей ---")
    regular_guest = Customer(name="Алексей", table_number=5)
    loyal_guest = LoyaltyCardHolder(name="Мария", table_number=12, points=1250)

    print(regular_guest)
    print(loyal_guest)

    # 3. Создаем чеки для обоих гостей с ОДИНАКОВЫМ набором блюд
    print("\n--- Формирование и расчет чеков ---")

    # Чек для обычного гостя (скидок нет)
    check_regular = Check(customer=regular_guest, items=menu_items)
    print(f"\n{check_regular}")
    print("Детализация: скидки не применены (гость не является держателем карты).")

    # Чек для лояльного гостя (скидка 15% только на блюда!)
    check_loyal = Check(customer=loyal_guest, items=menu_items)
    print(f"\n{check_loyal}")
    print("Детализация: применена скидка 15% ТОЛЬКО на блюда (Паста и Стейк). Напитки без скидки.")

    # Демонстрация работы полиморфизма "под капотом"
    print("\n--- Проверка полиморфизма (apply_discount) ---")
    print(f"Скидка на Пасту для лояльного гостя: {dish_pasta.apply_discount(True):.2f} руб.")
    print(f"Скидка на Вино для лояльного гостя: {drink_wine.apply_discount(True):.2f} руб. (напитки без скидки)")

if __name__ == "__main__":
    main()

ДОБРО ПОЖАЛОВАТЬ В РЕСТОРАН 'PYTHON GOURMET'

--- Формирование меню ---
Паста Карбонара | Цена: 650.00 руб. | Состав: Спагетти, бекон, сливки, пармезан
Стейк Рибай | Цена: 1800.00 руб. | Состав: Говядина, розмарин, чеснок
Вино красное | Цена: 450.00 руб. | Объем: 150 мл
Вода минеральная | Цена: 150.00 руб. | Объем: 500 мл

--- Регистрация гостей ---
Гость: Алексей | Столик: №5
Гость: Мария | Столик: №12 | Баллы лояльности: 1250

--- Формирование и расчет чеков ---

Чек для: Алексей | Позиций: 4 | Итого: 3050.00 руб.
Детализация: скидки не применены (гость не является держателем карты).

Чек для: Мария | Позиций: 4 | Итого: 2682.50 руб.
Детализация: применена скидка 15% ТОЛЬКО на блюда (Паста и Стейк). Напитки без скидки.

--- Проверка полиморфизма (apply_discount) ---
Скидка на Пасту для лояльного гостя: 97.50 руб.
Скидка на Вино для лояльного гостя: 0.00 руб. (напитки без скидки)
